In [1]:
import torch
from transformers import Wav2Vec2ForSequenceClassification

/Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
if torch.backends.mps.is_available():
    device = torch.device("mps")
    print("🍎 Using Apple Silicon GPU (MPS)")
else:
    device = torch.device("cpu")
    print("Using CPU")

🍎 Using Apple Silicon GPU (MPS)


In [3]:
model = Wav2Vec2ForSequenceClassification.from_pretrained(
    "facebook/wav2vec2-base",
    num_labels=2
)

Loading weights: 100%|██████████| 211/211 [00:00<00:00, 24004.51it/s]
[transformers] Wav2Vec2ForSequenceClassification LOAD REPORT from: facebook/wav2vec2-base
Key                          | Status     | 
-----------------------------+------------+-
project_q.bias               | UNEXPECTED | 
quantizer.weight_proj.weight | UNEXPECTED | 
project_hid.bias             | UNEXPECTED | 
project_hid.weight           | UNEXPECTED | 
quantizer.weight_proj.bias   | UNEXPECTED | 
quantizer.codevectors        | UNEXPECTED | 
project_q.weight             | UNEXPECTED | 
classifier.bias              | MISSING    | 
projector.weight             | MISSING    | 
projector.bias               | MISSING    | 
classifier.weight            | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [4]:
MODEL_PATH = "/Users/vjashwanth/Desktop/bestmodel.pth"

checkpoint = torch.load(
    MODEL_PATH,
    map_location=device
)

model.load_state_dict(
    checkpoint["model_state_dict"]
)

model.to(device)
model.eval()

print("✅ Your trained Deepfake Detection Model loaded!")

✅ Your trained Deepfake Detection Model loaded!


In [13]:
AUDIO_PATH = "/Users/vjashwanth/Desktop/speech.wav"
import librosa
import numpy as np
# Load audio exactly like training/testing
audio, sr = librosa.load(
    AUDIO_PATH,
    sr=16000,
    mono=True
)

# Convert to tensor
audio = torch.tensor(
    audio,
    dtype=torch.float32
)

# EXACTLY 4 seconds
max_length = 16000 * 4

audio = audio[:max_length]

if len(audio) < max_length:
    audio = torch.nn.functional.pad(
        audio,
        (0, max_length - len(audio))
    )

# Add batch dimension
input_values = audio.unsqueeze(0).to(device)

# Predict
with torch.no_grad():

    outputs = model(
        input_values=input_values
    )

    probabilities = torch.softmax(
        outputs.logits,
        dim=1
    )

    prediction = torch.argmax(
        probabilities,
        dim=1
    ).item()

    confidence = probabilities[0][prediction].item()

In [14]:
labels = {
    0: "FAKE 🚨",
    1: "REAL ✅"
}

print("\n🎙️ DEEPFAKE AUDIO DETECTION RESULT")
print("=" * 40)

print("Prediction:", labels[prediction])
print(f"Confidence: {confidence * 100:.2f}%")


🎙️ DEEPFAKE AUDIO DETECTION RESULT
Prediction: FAKE 🚨
Confidence: 95.37%
